# Association of infant sleep quality by the gut microbiota diversity & volatility

Summary: Modulation of infant sleep by the gut microbiota: sleep quality (babySQUID).

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from functions_script import load_tsv, scatterplot, plot_estimates_vertically

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# figures_path = "../figures"

In [ ]:
# create color palette
colors = [plt.cm.Spectral(i/float(6)) for i in range(7)]
timepoints_colors = [colors[0], colors[5], colors[6]]

## Load infants metadata

In [ ]:
# load infant metadata per age/timepoint
md_ages = load_tsv(f"{meta_data_path}/md_ages.tsv")
md_ages = md_ages.sort_values(by=["timepoint", "infant_id"])

print(md_ages.shape)
md_ages.head()

## Gut microbiota diversity & babySQUID

In [ ]:
# define data for diversity models
data_div = md_ages.dropna(subset=['observed_features', 'babySQUID']).copy()

In [ ]:
# linear mixed effects models for observed features and babySQUID
model_of = smf.mixedlm("babySQUID ~ observed_features + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_of.summary())

In [ ]:
# linear mixed effects models for shannon entropy and babySQUID
model_sh = smf.mixedlm("babySQUID ~ shannon_entropy + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_sh.summary())

In [ ]:
# linear mixed effects models for pielou evenness and babySQUID
model_pi = smf.mixedlm("babySQUID ~ pielou_evenness + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_pi.summary())

In [ ]:
# linear mixed effects models for faith pd and babySQUID
model_fa = smf.mixedlm("babySQUID ~ faith_pd + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_div, 
                    groups=data_div["infant_id"]).fit()
print(model_fa.summary())

In [ ]:
# scatterplots of age, shannon entropy and pielou evenness vs. babySQUID
fig, axs = plt.subplots(1, 3, figsize=(12, 6), sharey=True)

scatterplot(data_div, "age_days", "babySQUID", "timepoint", 
                    "", "babySQUID (0-1)", "Age (days)",
                    timepoints_colors, axs[0], loc_position = "lower right")

plt.subplots_adjust(wspace=1)

scatterplot(data_div, "shannon_entropy", "babySQUID", "timepoint", 
                    "", "Infant sleep quality index (0-1)", "Alpha diversity: Shannon entropy",
                    timepoints_colors, axs[1], loc_position = None)

scatterplot(data_div, "pielou_evenness", "babySQUID", "timepoint", 
                    "", "Infant sleep quality index (0-1)", "Alpha diversity: Pielou evenness",
                    timepoints_colors, axs[2], loc_position = None)

plt.tight_layout()

plt.savefig(f"{figures_path}/microbiome-and-sleep-sign.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots for shannon entropy and pielou evenness models
models = [model_sh, model_pi]
titles = ["babySQUID", ""]
plot_estimates_vertically(models, titles, f"{figures_path}/sleep_and_div_estimates_sign.pdf")

In [ ]:
# scatterplots of observed features and faith pd vs. babySQUID
fig, axs = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

scatterplot(data_div, "observed_features", "babySQUID", "timepoint", 
                    "", "babySQUID (0-1)", "Alpha diversity: Observed features",
                    timepoints_colors, axs[0], loc_position = "upper left")

scatterplot(data_div, "faith_pd", "babySQUID", "timepoint", 
                    "", "Infant sleep quality index (0-1)", 
                    "Alpha diversity: Faith phylogenetic diversity",
                    timepoints_colors, axs[1], loc_position = None)

plt.tight_layout()

plt.savefig(f"{figures_path}/microbiome-and-sleep-nonsign.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots for observed features and faith pd models
models = [model_of, model_fa]
plot_estimates_vertically(models, titles, f"{figures_path}/sleep_and_div_estimates_nonsign.pdf")

## Gut microbiota temporal volatility & babySQUID

In [ ]:
# define data for volatility models
data_vol = md_ages.dropna(subset=["volatility_braycurtis", "babySQUID"]).copy()

In [ ]:
# linear mixed effects models for bray-curtis volatility and babySQUID
model_bc = smf.mixedlm("babySQUID ~ volatility_braycurtis + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_vol, 
                    groups=data_vol["infant_id"]).fit()
print(model_bc.summary())

In [ ]:
# linear mixed effects models for jaccard volatility and babySQUID
model_ja = smf.mixedlm("babySQUID ~ volatility_jaccard + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_vol, 
                    groups=data_vol["infant_id"]).fit()
print(model_ja.summary())

In [ ]:
# linear mixed effects models for unweighted unifrac volatility and babySQUID
model_uu = smf.mixedlm("babySQUID ~ volatility_unweighted_unifrac + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_vol,
                    groups=data_vol["infant_id"]).fit()
print(model_uu.summary())

In [ ]:
# linear mixed effects models for weighted unifrac volatility and babySQUID
model_wu = smf.mixedlm("babySQUID ~ volatility_weighted_unifrac + BCQ_Attunement + ASQ_Composite + age_days + sex",
                    data_vol, 
                    groups=data_vol["infant_id"]).fit()
print(model_wu.summary())

In [ ]:
# scatterplots of volatility metrics vs. babySQUID
fig, axs = plt.subplots(1, 4, figsize=(16, 6), sharey=True)

scatterplot(data_vol, "volatility_braycurtis", "babySQUID", "timepoint", 
                    "", "babySQUID (0-1)", "Temporal volatility: \n Bray-Curtis dissimilarity",
                    timepoints_colors, axs[0], loc_position = "upper right")

scatterplot(data_vol, "volatility_jaccard", "babySQUID", "timepoint", 
                    "", "babySQUID (0-1)", "Temporal volatility: \n Jaccard similarity index",
                    timepoints_colors, axs[1], loc_position = None)

scatterplot(data_vol, "volatility_unweighted_unifrac", "babySQUID", "timepoint", 
                    "", "babySQUID (0-1)", "Temporal volatility: \n Unweighted UniFrac distance",
                    timepoints_colors, axs[2], loc_position = None)

scatterplot(data_vol, "volatility_weighted_unifrac", "babySQUID", "timepoint", 
                    "", "babySQUID (0-1)", "Temporal volatility: \n Weighted UniFrac distance",
                    timepoints_colors, axs[3], loc_position = None)

plt.tight_layout()

plt.savefig(f"{figures_path}/volatility-and-sleep.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# coefficient plots for volatility models
models = [model_bc, model_ja, model_uu, model_wu]
titles = ["babySQUID", "", "", ""]
plot_estimates_vertically(models, titles, f"{figures_path}/sleep_and_vol_estimates.pdf")